In [ ]:
using Pkg
Pkg.activate(".")
Pkg.develop(path="..")

using Revise

In [ ]:
isCuda = try
    success(`nvidia-smi`)
catch
    false
end

In [ ]:
if isCuda
    println("CUDA is available. Loading CUDA.jl...")
    using CUDA
end

In [ ]:
using bslLD, Plots, Statistics

    
isCuda &&bslLD.use_cuda!()

In [ ]:
mutable struct Diag
    rhoe::Vector
    rhoi::Vector
    fi ::Vector
    fe ::Vector
end
Diag() = Diag([], [], [], [])

function diags!(diags, fe, fi, rhoe, rhoi, grid)
    push!(diags.rhoe, copy(rhoe.data[:]))
    push!(diags.rhoi, copy(rhoi.data[:]))
    push!(diags.fi, copy(fi.data))
    push!(diags.fe, copy(fe.data))
end

function diags!(diags,fi , grid)
    push!(diags.fi, copy(fi.data))
    push!(diags.rhoi, copy(bslLD.compute_density(fi, grid).data[:]))
end

function step!(fi, fe, grid, simTime)
    rhoe = bslLD.compute_density(fe, grid)
    rhoi = bslLD.compute_density(fi, grid)
    sol = bslLD.solve_fields(bslLD.Moments(rhoi-rhoe), grid, bslLD.PoissonFieldSolver(-1.0))
    simTime.fraction_dt = 0.5
    bslLD.advectV!(fe, grid, simTime, sol.E)
    bslLD.advectV!(fi, grid, simTime, sol.E)
    simTime.fraction_dt = 1.0
    bslLD.advectX!(fe, grid, simTime)
    bslLD.advectX!(fi, grid, simTime)
    sol = bslLD.solve_fields(bslLD.Moments(rhoi-rhoe), grid, bslLD.PoissonFieldSolver(-1.0))
    simTime.fraction_dt = 0.5
    bslLD.advectV!(fe, grid, simTime, sol.E)
    bslLD.advectV!(fi, grid, simTime, sol.E)

    simTime.fraction_dt = 1.0
    return rhoe, rhoi
end


function step!(fi, grid, simTime)
    rhoi = bslLD.compute_density(fi, grid)
    sol = bslLD.solve_fields(bslLD.Moments(rhoi), grid, bslLD.PoissonFieldSolver(-1.0))
    sol.E[1].data .= 1.0
    sol.E[2].data .= 0.0
    simTime.fraction_dt = 0.5
    bslLD.advectV!(fi, grid, simTime, sol.E)
    simTime.fraction_dt = 1.0
    bslLD.advectX!(fi, grid, simTime)
    sol = bslLD.solve_fields(bslLD.Moments(rhoi), grid, bslLD.PoissonFieldSolver(-1.0))
    sol.E[1].data .= 1.0
    sol.E[2].data .= 0.0
    simTime.fraction_dt = 0.5
    bslLD.advectV!(fi, grid, simTime, sol.E)

    simTime.fraction_dt = 1.0
    return rhoi
end


In [ ]:
# k = (kx, 0, 0),  B0 = z-hat  (grid.Bdir = 3)
# Only Bz and (Ex, Ey) evolve; Ez = 0 and Pi_diff_z = 0 exactly for this geometry.

beta_i  = 0.1     # ion beta
mu      = 100.0    # mass ratio m_i / m_e
epsilon = 1e-6     # perturbation amplitude

Lx   = 20.0
Nx   = 8
Nv   = 64
vmax = 8.0

grid    = bslLD.Grid([0.0, -vmax, -vmax], [Lx, vmax, vmax], [Nx, Nv, Nv], 1, 1.0, 3)
simTime = bslLD.SimulationTime(0.005, 10.0; gyro_frequency=1.0)

# Ion distribution: Maxwellian in (vx, vy) with a random density perturbation in x
initFuncx(x) = 1.0 + epsilon * exp(-(x - Lx/2)^2 / (Lx/10)^2)
initFuncv(v) = exp(-(v)^2 / 2) / sqrt(2pi)

f_i = bslLD.Distribution(grid, 0.0;
    initFuncv = initFuncv,
    q=1
)

f_e = bslLD.Distribution(grid, 0.0;
    initFuncv = initFuncv,
    q=-1
)




In [ ]:
diags_i = Diag()
diags_e = Diag()
while bslLD.continue_advection(simTime,true)
    step!(f_i, grid, simTime)
    step!(f_e, grid, simTime)
    diags!(diags_i, f_i, grid)
    diags!(diags_e, f_e, grid)
    bslLD.advance!(simTime)
end    

In [ ]:
fdiag_i = x-> bslLD.DistributionGrid1d2v{Float64,bslLD.Cart,typeof(x)}(x,1.0,1.0)
fdiag_e = x-> bslLD.DistributionGrid1d2v{Float64,bslLD.Cart,typeof(x)}(x,1.0,-1.0)

In [ ]:
jAnaq(t0, q) = [sin(q * t0), -1 + cos(q * t0)]
ΠAnaq(t0, q) = [(3 - cos(2 * q * t0))/2  (-1 + cos(q * t0)) * sin(q * t0); 
               (-1 + cos(q * t0)) * sin(q * t0)  2 + (-2 + cos(q * t0)) * cos(q * t0)]

jAnaSelect(q) = t-> jAnaq(q,t)
ΠAnaSelect(q) = t-> ΠAnaq(q,t)

In [ ]:
j_ilab = [map(x->mean(x.data, dims=1)[1],bslLD.compute_current(fdiag_i(diags_i.fi[i]),grid, simTime.gyro_frequency * simTime.dt*i)) for i in 1:length(diags_i.fi) ]
plot(collect(simTime)[2:end],transpose(hcat(j_ilab...)))

J_i = hcat([[jAnaSelect(1)(x)[1], jAnaSelect(1)(x)[2]] for x in simTime]...)';
plot!(collect(simTime), J_i, label=["j₁" "j₂"])


In [ ]:
j_elab = [map(x->mean(x.data, dims=1)[1],bslLD.compute_current(fdiag_e(diags_e.fi[i]),grid, simTime.gyro_frequency * simTime.dt*i)) for i in 1:length(diags_e.fi) ]
plot(collect(simTime)[2:end], transpose(hcat(j_elab...)))

J_e = hcat([[jAnaSelect(-1)(x)[1], jAnaSelect(-1)(x)[2]] for x in simTime]...)';
plot!(collect(simTime), J_e, label=["j₁" "j₂"])

In [ ]:
pi_lab = [map(x->mean(x.data, dims=1)[1],bslLD.compute_momentum_tensor(fdiag_i(diags_i.fi[i]),grid, simTime.gyro_frequency * simTime.dt*i)) for i in 1:length(diags_i.fi) ]
plot(collect(simTime)[2:end],transpose(hcat(pi_lab...)))

ΠAna_i = ΠAnaSelect(1)

M_i = hcat([[ΠAna_i(x)[1,1], ΠAna_i(x)[1,2], ΠAna_i(x)[2,2]] for x in simTime]...)';
plot!(collect(simTime), M_i, label=["P₁₁" "P₁₂" "P₂₂"], linestyle=:dash)


In [ ]:
pi_lab = [map(x->mean(x.data, dims=1)[1],bslLD.compute_momentum_tensor(fdiag_e(diags_e.fi[i]),grid, simTime.gyro_frequency * simTime.dt*i)) for i in 1:length(diags_e.fi) ]
plot(collect(simTime)[2:end],transpose(hcat(pi_lab...)))

ΠAna_e = ΠAnaSelect(-1)

M_e = hcat([[ΠAna_e(x)[1,1], ΠAna_e(x)[1,2], ΠAna_e(x)[2,2]] for x in simTime]...)';
plot!(collect(simTime), M_e, label=["P₁₁" "P₁₂" "P₂₂"], linestyle=:dash)

In [ ]:
## Plot error between numerical and analytical solutions for j and Π

error_j_i = [norm(map(x->mean(x.data, dims=1)[1],bslLD.compute_current(fdiag_i(diags_i.fi[i]),grid, simTime.gyro_frequency * simTime.dt*i)) - jAnaSelect(1)(simTime.dt*i)) for i in 1:length(diags_i.fi) ]

plot(collect(simTime)[2:end], error_j_i, label="Error in j_i", yaxis=:log)

error_j_e = [norm(map(x->mean(x.data, dims=1)[1],bslLD.compute_current(fdiag_e(diags_e.fi[i]),grid, simTime.gyro_frequency * simTime.dt*i)) - jAnaSelect(-1)(simTime.dt*i)) for i in 1:length(diags_e.fi) ]

plot!(collect(simTime)[2:end], error_j_e, label="Error in j_e", yaxis=:log)



In [ ]:
locData = transpose(hcat(map(x-> x.-mean(x), Array.(diags.rho))...))


Nx, Ny = size(locData)
w = kaiser(Ny, 6)

windowed = locData .* w'        # broadcast along second dim (1 × Ny)

heatmap(log.(abs.(fft(windowed))[1:200,1:round(Int,Ny/2)]))

In [ ]:

function measureStep(backendFunc,grid)
    backendFunc()

    f = bslLD.Distribution(grid, 0.5,initFuncv=initFuncv, initFuncx=initFuncx);
    e = bslLD.empty_vectorfield(grid);

    [step!(f, grid, Diag()) for _ in 1:10]   

    @time [step!(f, grid, Diag()) for _ in 1:10]
end

grid =  bslLD.Grid([0.0,-4.0,-4.0],[60.0,4.0,4.0],[256,257,257],0.02,10000,1, 1.0, 1)

measureStep(() -> bslLD.use_cuda!(),grid)
measureStep(() -> bslLD.use_cpu!(), grid)


#   1.270276 seconds (278.26 k allocations: 8.469 MiB, 1.77% gc time, 64.35% compilation time)
# 247.798143 seconds (46.79 k allocations: 29.209 GiB, 84.31% gc time, 0.89% compilation time)
#   0.059636 seconds (60.53 k allocations: 2.844 MiB, 20.65% gc time, 61.27% compilation time)
#   0.126698 seconds (46.27 k allocations: 83.402 MiB, 28.70% compilation time)





In [ ]:
#   0.018241 seconds (22.21 k allocations: 1.417 MiB, 22.05% gc time)
#  16.719033 seconds (2.88 k allocations: 1.254 GiB, 88.55% gc time)


grid =  bslLD.Grid([0.0,-4.0],[60.0,4.0],[1024,1024],0.02,10000,1, 1.0, 1)

measureStep(() -> bslLD.use_cuda!(),grid)
measureStep(() -> bslLD.use_cpu!(), grid)



In [ ]:
grid =  bslLD.Grid([0.0,-4.0,-4.0],[60.0,4.0,4.0],[256,255,255],0.02,10000,1, 1.0, 1)
f = bslLD.Distribution(grid, 0.5);

plan = bslLD.AdvectionPlan(f, grid)


In [ ]:
size(f.data)

In [ ]:
bslLD.use_cpu!()

In [ ]:
function compare_allocations(backendFunc)
    backendFunc()
    grid =  bslLD.Grid([0.0,-4.0,-4.0],[60.0,4.0,4.0],[256,257,257],0.02,10000,1, 1.0, 1)
    f = bslLD.Distribution(grid, 0.5);
    e = bslLD.empty_vectorfield(grid);

    plan = bslLD.AdvectionPlan(f, grid)

    #Warmup
    bslLD.advectX!(f,grid, plan)
    bslLD.advectX!(f,grid)
    bslLD.advectV!(f,grid,e)
    bslLD.advectV!(f,grid,e, plan)

    println("Measuring allocations for ", typeof(f.data))
    println("AdvectX with plan:")
    @time bslLD.advectX!(f,grid, plan)
    println("AdvectX without plan:")
    @time bslLD.advectX!(f,grid)
    println("AdvectV with plan:")
    @time bslLD.advectV!(f,grid,e, plan)
    println("AdvectV without plan:")
    @time bslLD.advectV!(f,grid,e)

    println(" ")

    println("Check for type stability:")
    println("AdvectX with plan:")
    @code_warntype bslLD.advectX!(f,grid, plan)
    println("AdvectX without plan:")
    @code_warntype bslLD.advectX!(f,grid)
    println("AdvectV with plan:")
    @code_warntype bslLD.advectV!(f,grid,e, plan)
    println("AdvectV without plan:")
    @code_warntype bslLD.advectV!(f,grid,e)
end

In [ ]:
compare_allocations(() -> bslLD.use_cuda!())


In [ ]:
compare_allocations(() -> bslLD.use_cpu!())

In [ ]:
CUDA.@profile bslLD.advectX!(f, grid, plan)

In [ ]:
CUDA.@profile bslLD.advectV!(f, grid, e, plan)

In [ ]:
print("Size of f.data: ")
println(size(f.data))
CUDA.@profile bslLD.advectV!(f, grid, e, plan)

In [ ]:

bslLD.use_cuda!()

grid =  bslLD.Grid([0.0,-4.0,-4.0],[60.0,4.0,4.0],[256,257,257],0.02,10000,1, 1.0, 1)
f = bslLD.Distribution(grid, 0.5);
e = bslLD.empty_vectorfield(grid);

plan = bslLD.AdvectionPlan(f, grid)


@code_warntype bslLD._advect_x_planned!(f, grid, plan, CUDABackend())
@code_warntype bslLD._advect_v_planned!(f, grid, e, plan, CUDABackend())

In [ ]:
@allocated ntuple(d -> e[d].data, Val(2))
#6000

In [ ]:
arr = f.data

In [ ]:
@allocated kernel!(arr, ctx; ndrange=length(arr))

In [ ]:

bslLD.use_cpu!()

grid =  bslLD.Grid([0.0,-4.0,-4.0],[60.0,4.0,4.0],[256,257,257],0.02,10000,1, 1.0, 1)
f = bslLD.Distribution(grid, 0.5);
e = bslLD.empty_vectorfield(grid);

plan = bslLD.AdvectionPlan(f, grid)


@code_warntype bslLD._advect_x_planned!(f, grid, plan, bslLD.KernelAbstractions.CPU())
@code_warntype bslLD._advect_v_planned!(f, grid, e, plan, bslLD.KernelAbstractions.CPU())

In [ ]:
plan.backend